# YouTube Data Scraper
### Automated channel and video analytics using the YouTube Data API v3

Supports multiple channels. Data is stored in a local SQLite database (`youtube.db`) with two tables — `channels` and `videos`. Re-running upserts rows so there are no duplicates.

**Setup:** copy `.env.example` to `.env` and fill in your `YOUTUBE_API_KEY` and `YOUTUBE_CHANNEL_IDS` (comma-separated).

---

## 1. Imports

In [ ]:
import os
import sys

import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
from src.scraper import get_channel_stats, get_video_ids, get_video_details
from src.db import get_engine, save_channels, save_videos

load_dotenv()

## 2. Configuration

In [ ]:
channel_ids = [cid.strip() for cid in os.getenv('YOUTUBE_CHANNEL_IDS', '').split(',') if cid.strip()]
print(f'{len(channel_ids)} channel(s) to scrape: {channel_ids}')

## 3. Scrape All Channels

In [ ]:
all_channel_stats = []
all_video_details = []

for channel_id in channel_ids:
    stats = get_channel_stats(channel_id)
    all_channel_stats.append(stats)

    video_ids = get_video_ids(channel_id)
    print(f"{stats['title_channel']}: {len(video_ids)} videos")

    details = get_video_details(video_ids)
    for d in details:
        d['channel_id'] = channel_id
    all_video_details.extend(details)

## 4. Save to Database

Upserts into `youtube.db` — safe to re-run, no duplicate rows.

In [ ]:
engine = get_engine()
save_channels(engine, all_channel_stats)
save_videos(engine, all_video_details)
print(f'Saved {len(all_channel_stats)} channel(s) and {len(all_video_details)} video(s) to youtube.db')

## 5. Preview

In [ ]:
import pandas as pd
pd.read_sql('SELECT * FROM channels', engine)

In [ ]:
pd.read_sql('SELECT channel_id, title_video, view_count, like_count FROM videos LIMIT 20', engine)